# Stage 3 — 医療ガイドラインRAG（検索→根拠づけ→回答）

このノートは medguide-rag プロジェクトの **Stage 3**（RAG = 検索拡張生成）の成果物です。
題材は **WHO HEARTS「Healthy-lifestyle counselling」**（生活習慣カウンセリング・全30ページ・英語）。
この文書だけを知識源に、日本語の質問へ**文書を根拠にした日本語の回答**を返す仕組みを作ります。

> **このノートの読み方（Pythonが分からなくても妥当性を判断できる工夫）**
> 各質問への回答の前に**出典追跡パネル**（検索でヒットした文書箇所＝出典ページと類似度の表）を出します。
> 回答文の主張には `[1]` のような番号が付き、パネルの行と対応します。**回答が実在の文書箇所に
> 根拠づいているか（＝作り話＝幻覚でないか）**を、コードを読まずに目視で確認できます。
> これは Stage 1・2 の「定説照合」を、RAG向けに「**回答根拠の出典追跡**」へ発展させたものです。

> **ライセンスと注意**: 題材文書は WHO, HEARTS technical package: Healthy-lifestyle counselling (2018)。
> © World Health Organization 2018, licensed under **CC BY-NC-SA 3.0 IGO**（非営利での再配布・翻訳可）。
> 本ノートは学習用デモで、回答は文書の要約・引用であり**医療上の助言ではありません**。
> 英語文書を根拠に日本語で回答します（RAGのクロスリンガル能力の実演）。

## このノートで通す「RAGの流れ」

| 段階 | やること | 主に使う道具 |
|---|---|---|
| ① 準備 | PDFを読み、検索しやすい単位（チャンク）に分割 | pypdf / RecursiveCharacterTextSplitter |
| ② 索引 | チャンクをベクトル化し Chroma に格納 | multilingual-e5 / Chroma |
| ③ 検索 | 質問に近いチャンクを取り出す | ベクトル類似度検索 |
| ④ 生成 | 取り出した文脈だけを根拠に回答 | LangGraph + Claude |
| ⑤ 照合 | 回答の根拠を出典追跡パネルで目視 | （幻覚でないかの健全性チェック） |


## 準備 — モジュールの読み込み

このノートは `notebooks/` で実行される前提です。プロジェクト直下を import パスに加え、
実装本体（`src/ingest`・`src/rag`）を呼び出します。パイプラインの各段はこのモジュールに実装済みで、
ノートでは「流れを追う」ことに集中します。


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # プロジェクト直下を import パスに追加

import pandas as pd
from src.ingest import load_pdf, load_and_chunk
from src.rag import get_or_build_index, make_rag, ask, DEFAULT_PDF, EMBED_MODEL, ANTHROPIC_MODEL, TOP_K

print("題材文書:", DEFAULT_PDF.name)
print("埋め込みモデル:", EMBED_MODEL)
print("生成モデル:", ANTHROPIC_MODEL)

## ① データ準備 — PDFを読み、チャンクに分割する

RAGはまず文書を「検索しやすい小さな単位（チャンク）」に切り分けます。長い文書のどこに答えがあるか
分からないため、段落など**意味の切れ目**を優先して 1000 文字程度に分割し、各チャンクに
**出典（ファイル名・ページ）**を持たせます。この出典が後の「出典追跡」に効きます。


In [ ]:
pages = load_pdf(DEFAULT_PDF)
chunks = load_and_chunk(DEFAULT_PDF)
print(f"ページ数: {len(pages)} → チャンク数: {len(chunks)}")

# 1チャンクの中身と出典メタデータを確認
ex = chunks[10]
print("\n--- チャンク例（metadata と本文先頭）---")
print("出典:", ex.metadata)
print(ex.page_content[:220], "...")

## ② 索引 — チャンクをベクトル化して Chroma に格納する

各チャンクを**埋め込み（意味を表す数値ベクトル）**に変換し、ベクトルDB **Chroma** に貯めます。
埋め込みは多言語モデル **multilingual-e5**（ローカル実行・APIキー不要）を使うため、
**日本語の質問と英語の文書を同じ意味空間で比較**でき、言語をまたいだ検索ができます。

初回は索引を構築し `chroma/` に保存します（2回目以降は再利用）。


In [ ]:
vs, n = get_or_build_index()   # 無ければ構築・あれば読み込み
print(f"検索対象の索引: {n} チャンク（保存先 chroma/）")

## ③ 検索デモ — 質問に近い文書箇所を取り出す（生成の前段）

回答を作る前に、まず「質問に意味が近いチャンク」を取り出します。ここは **LLMを使わない純粋な検索**で、
日本語の質問に対して英語文書のどこがヒットするか（＝クロスリンガル検索）を確認できます。
数値は類似度（1に近いほど近い）です。


In [ ]:
q_demo = "運動はどのくらい行うべきですか？"
hits = vs.similarity_search_with_relevance_scores(q_demo, k=TOP_K)

rows = []
for i, (d, score) in enumerate(hits, start=1):
    rows.append({
        "番号": i,
        "出典ページ": f"p.{d.metadata.get('page')}",
        "類似度": round(float(score), 3),
        "抜粋（先頭）": " ".join(d.page_content.split())[:70] + "…",
    })
print("質問:", q_demo)
pd.DataFrame(rows)

## ④ 検索→生成を1フローに（LangGraph）

検索（retrieve）と回答生成（generate）を、状態を持つ小さなグラフ **LangGraph** の2ノードにまとめます。
`retrieve` が取り出したチャンク（出典付き）を状態に載せ、`generate` がその**文脈だけを根拠に**
Claude で日本語の回答を作ります。プロンプトで「文脈に無いことは『記載なし』と答える」ことを強制し、
作り話（幻覚）を抑えます。

> ここから先は回答生成のため **`.env` の `ANTHROPIC_API_KEY`** が必要です。


In [ ]:
# 索引（構築済み）＋ LangGraph をまとめて用意
graph, vs, n = make_rag()

def show_rag(question):
    """質問→出典追跡パネル（表）→日本語回答 を表示する。"""
    state = ask(graph, question)
    rows = []
    for i, (d, score) in enumerate(state["docs"], start=1):
        rows.append({
            "番号": i,
            "出典ページ": f"p.{d.metadata.get('page')}",
            "類似度": round(float(score), 3),
            "抜粋（根拠箇所の先頭）": " ".join(d.page_content.split())[:70] + "…",
        })
    print("【質問】", question)
    print("\n【出典追跡パネル】回答はこの検索結果だけを根拠にしています:")
    display(pd.DataFrame(rows))
    print("\n【回答（日本語・[n]は上の番号に対応）】")
    print(state["answer"])
    return state

### 質問例1 — 身体活動（運動）の推奨

回答の各主張に付く `[n]` が、上の出典追跡パネルのどの行（＝文書のどのページ）に基づくかを表します。
パネルの抜粋と回答の向きが一致していれば、回答は実在の文書に根拠づいており健全と判断できます。


In [ ]:
_ = show_rag("成人は身体活動（運動）をどのくらい行うことが推奨されていますか？")

### 質問例2 — 禁煙・タバコに関する助言

別の話題でも、検索→根拠づけ→回答の同じ流れで、出典付きの回答が得られることを確認します。


In [ ]:
_ = show_rag("タバコをやめるために、どのような助言が示されていますか？")

## ⑤ 健全性チェック — 「文書に無いこと」は答えない（幻覚の抑止）

RAGの価値は「知らないことを作り話しない」ことにあります。題材文書（生活習慣カウンセリング）に
**書かれていない**質問を投げ、モデルが「提供された文書には記載がありません」と正直に答えることを確認します。
これができると、回答を安心して根拠に使えます。


In [ ]:
_ = show_rag("糖尿病の患者に処方するインスリンの具体的な投与量は何単位ですか？")

## まとめ — 分かったことと次への橋渡し

RAGの基本的な流れ（準備 → 索引 → 検索 → 生成 → 出典追跡）を一通り通しました。

### やったこと
1. **準備**（①）: WHO HEARTS の PDF を出典付きチャンクに分割
2. **索引**（②）: 多言語埋め込みで Chroma に格納（日本語質問→英語文書のクロスリンガル検索）
3. **検索**（③）: 質問に近い文書箇所を類似度で取り出す
4. **生成**（④）: LangGraph で「文脈のみを根拠に」Claude が日本語回答（引用付き）
5. **出典追跡**（⑤）: 回答の根拠をパネルで目視。文書に無い質問は「記載なし」と答え、幻覚を抑止

### 次のステージへ
Stage 4 では、この RAG の**回答品質を自動評価**します（LLM-as-judge）。「回答が根拠に忠実か／
質問に答えているか／誤りが無いか」を複数視点で採点する評価ループを組み、
本ノートで確立した「出典追跡＝根拠づけの透明化」が評価の土台になります。
